### Step 1: Import Required Libraries
Import all the necessary libraries for web scraping:
- `requests`: For making HTTP requests
- `BeautifulSoup`: For parsing HTML
- `urljoin`: For constructing absolute URLs
- `pandas`: For data manipulation and export
- `os`: For file system operations

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import os

### Step 2: Scrape Books from Multiple Pages
- Create the `images` folder if it doesn't exist
- Loop through the first 3 pages of the book catalogue
- Extract book details: title, price, rating, and image
- Download and save each book's image to the `images/` folder
- Store all book data in a list of dictionaries

In [2]:
# Create images folder if it doesn't exist
os.makedirs("images", exist_ok=True)

base_url= "https://books.toscrape.com/catalogue/page-1.html"
raw_books = []
current_page = 1
url = base_url

while url and current_page < 5:
    resopnse = requests.get(url)

    soup = BeautifulSoup(resopnse.text, "html.parser")

    articles = soup.find_all("article", class_="product_pod")
    for article in articles:
        title = article.h3.a["title"]
        price = article.find("p", class_="price_color").text[2:]
        rating = article.find("p", class_="star-rating")["class"][1]
        imag_src = article.find("img")["src"]
        # image= urljoin(base_url, imag_src)
        image_url = "https://books.toscrape.com/" + imag_src
        
        img_content = requests.get(image_url).content
        with open("images/" + title[:50].replace(":", "").replace("#", "") + ".jpg", "wb") as f:
            f.write(img_content)
        
        raw_books.append({
            "title": title,
            "price": price,
            "rating": rating,
            "file_path": f"images/{title[:50].replace(':', '').replace('#', '')}.jpg"
        })

        print("name" , title)
        print("price" , price)
        print("rating" , rating)
        print("image_url" , image_url)
        print("file_path" , f"images/{title[:50].replace(':', '').replace('#', '')}.jpg")
        print("-------------")

    next_page = soup.find("li", class_="next")
    current_page += 1
    url = f"https://books.toscrape.com/catalogue/page-{current_page}.html" if next_page else None

name A Light in the Attic
price 51.77
rating Three
image_url https://books.toscrape.com/../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg
file_path images/A Light in the Attic.jpg
-------------
name Tipping the Velvet
price 53.74
rating One
image_url https://books.toscrape.com/../media/cache/26/0c/260c6ae16bce31c8f8c95daddd9f4a1c.jpg
file_path images/Tipping the Velvet.jpg
-------------
name Soumission
price 50.10
rating One
image_url https://books.toscrape.com/../media/cache/3e/ef/3eef99c9d9adef34639f510662022830.jpg
file_path images/Soumission.jpg
-------------
name Sharp Objects
price 47.82
rating Four
image_url https://books.toscrape.com/../media/cache/32/51/3251cf3a3412f53f339e42cac2134093.jpg
file_path images/Sharp Objects.jpg
-------------
name Sapiens: A Brief History of Humankind
price 54.23
rating Five
image_url https://books.toscrape.com/../media/cache/be/a5/bea5697f2534a2f86a3ef27b5a8c12a6.jpg
file_path images/Sapiens A Brief History of Humankind.jpg
-------------
n

### Step 3: Convert to DataFrame
Convert the scraped data into a pandas DataFrame for easier analysis and view the first few rows.

In [3]:
df = pd.DataFrame(raw_books)
df.head()

,title,price,rating,file_path
0,A Light in the Attic,51.77,Three,images/A Light in the Attic.jpg
1,Tipping the Velvet,53.74,One,images/Tipping the Velvet.jpg
2,Soumission,50.10,One,images/Soumission.jpg
3,Sharp Objects,47.82,Four,images/Sharp Objects.jpg
4,Sapiens: A Brief History of Humankind,54.23,Five,images/Sapiens A Brief History of Humankind.jpg


### Step 4: Export to CSV
Save the scraped book data to a CSV file named `scraped_books.csv`.

In [4]:
df.to_csv("scraped_books.csv", index=False)